# 09 — Ensemble & Submission

Run this **last**, after any subset of notebooks 02–08 have produced their `oof_*.csv` / `test_pred_*.csv`
files. It only loads whichever of those files it finds, so you don't need to run all 7 models — it'll blend
whatever is available.

Uses **greedy forward selection** (Caruana et al., 2004) to pick blend weights: starting from an empty
ensemble, it repeatedly adds whichever model (allowing repeats) most improves the running average's OOF
AUC, stopping when nothing helps anymore. This is a robust way to blend an arbitrary number of models
directly against the competition metric — it typically beats both the best single model and a naive
equal-weight average.

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
from sklearn.metrics import roc_auc_score

DATA_DIR = "."   # <-- folder with train_features.csv and all oof_*.csv / test_pred_*.csv files

train_fe = pd.read_csv(f"{DATA_DIR}/train_features.csv")
y_df = train_fe[['id', 'addicted_label']]

oof_files = sorted(glob.glob(f"{DATA_DIR}/oof_*.csv"))
test_files = sorted(glob.glob(f"{DATA_DIR}/test_pred_*.csv"))
print("found OOF files:", [os.path.basename(f) for f in oof_files])
print("found test files:", [os.path.basename(f) for f in test_files])


In [ ]:
oof_dict = {}
for f in oof_files:
    name = os.path.basename(f).replace("oof_", "").replace(".csv", "")
    df = pd.read_csv(f).merge(y_df, on='id', how='right')
    oof_dict[name] = df.sort_values('id')['oof_pred'].values

y = y_df.sort_values('id')['addicted_label'].values

test_dict = {}
test_ids = None
for f in test_files:
    name = os.path.basename(f).replace("test_pred_", "").replace(".csv", "")
    df = pd.read_csv(f).sort_values('id')
    test_dict[name] = df['test_pred'].values
    test_ids = df['id'].values

for name, oof in oof_dict.items():
    print(f"{name:5s} solo OOF AUC: {roc_auc_score(y, oof):.5f}")


In [ ]:
def greedy_ensemble(oof_dict, y, max_iters=60, tol=1e-6):
    names = list(oof_dict.keys())
    selected = []
    current_sum = np.zeros(len(y))
    best_auc = -1
    for i in range(max_iters):
        best_name, best_round_auc = None, -1
        for name in names:
            trial = (current_sum + oof_dict[name]) / (len(selected) + 1)
            auc = roc_auc_score(y, trial)
            if auc > best_round_auc:
                best_round_auc, best_name = auc, name
        if best_round_auc <= best_auc + tol and len(selected) > 0:
            break
        selected.append(best_name)
        current_sum += oof_dict[best_name]
        best_auc = best_round_auc
    counts = {n: selected.count(n) for n in set(selected)}
    weights = {n: c / len(selected) for n, c in counts.items()}
    return weights, best_auc, selected

weights, best_auc, selection_order = greedy_ensemble(oof_dict, y)
print("Greedy blend weights:", {k: round(v, 3) for k, v in weights.items()})
print("Greedy blended OOF AUC:", best_auc)


In [ ]:
final_test_pred = sum(w * test_dict[name] for name, w in weights.items())

submission = pd.DataFrame({
    'id': test_ids,
    'addicted_label': final_test_pred
})
submission.to_csv(f"{DATA_DIR}/submission.csv", index=False)
submission.head()


## Ideas if you want to push further

- **Optuna/hyperparameter search** on LightGBM/XGBoost/CatBoost — the configs in those notebooks are
  reasonable defaults, not tuned. `num_leaves`, `max_depth`, `min_child_samples`/`min_child_weight`, and
  `reg_lambda` are the highest-leverage knobs for this kind of tabular data.
- **More seeds per model, averaged** — rerun a model notebook with a different `SEED` and average the
  resulting OOF/test predictions with the first run before blending; cheap variance reduction.
- **Stacking** — instead of a weighted average, fit a simple logistic regression (or another light model)
  on the OOF columns as meta-features, rather than the greedy blend used here.
- **Target/frequency encoding** for the categorical columns computed out-of-fold, even though they showed
  near-zero marginal signal in isolation — GBMs can sometimes still extract small conditional interactions.
- **Pseudo-labeling** — add high-confidence test predictions (e.g., > 0.99 or < 0.01) back into training
  data for a second round of fitting, common in Playground-series solutions.
- **Deeper/wider NN, or a TabNet/FT-Transformer style architecture** — the simple MLP in notebook 08 is a
  baseline; a better-tuned NN (more epochs, learning-rate schedule, wider embeddings) could add more
  diversity value.
